# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Primary operational task: ranking / scoring inside a sequential hybrid ML system.**

The decision to automate is: **which content pages should receive attention first, why, and what action should be considered?** Because the final output is an ordered intervention queue, ranking/scoring is the primary task. The ranking is deliberately built from earlier ML stages rather than from a manually weighted rule.

The methodology is a dependency chain:

1. **Signal analysis** builds a leakage-safe pre-decision feature set from observable search, traffic, engagement, freshness, and content measurements.
2. **Clustering** groups pages into empirical performance archetypes. Its purpose is not decorative segmentation: it establishes peer groups and local comparison baselines.
3. **Archetype-relative analysis** measures how unusual each page is relative to similar pages, for example whether its CTR, engagement, freshness, or visibility is weak for its peer group rather than merely weak in absolute terms.
4. **Classification / prediction** estimates a future observed performance state using only information available before the decision point, including page-level and archetype-relative information.
5. **Impact estimation** combines predicted future risk with the expected size of adverse movement and measured page exposure/value.
6. **Ranking / scoring** orders eligible pages by expected adverse impact, preferably within client so large clients do not automatically dominate.
7. **Action generation** uses the page archetype, predicted direction, and strongest abnormal signals to produce reason codes and an evidence-based suggested action.

In shorthand:

**observable signals → archetype / peer context → relative abnormalities → future-state prediction → expected impact → ranked priority → suggested action**

The suggested action is a **testable intervention hypothesis**, not an assumed causal truth. The system can motivate why an action should be considered from the observed pattern and predicted risk. Predictive validity can be tested retrospectively; whether the intervention itself causes improvement can later be tested prospectively with a controlled experiment or another valid causal design.

In [ ]:
import pandas as pd
from pathlib import Path

candidate_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find data/raw/content_refresh_anonymized.csv")

df = pd.read_csv(data_path)

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Pseudonymized clients: {df['client_id'].nunique():,}")

assert df['content_id'].nunique() == len(df), (
    "Expected one row per pseudonymized content item."
)

methodology = [
    "signal analysis",
    "clustering / peer context",
    "archetype-relative deviations",
    "future-state prediction",
    "impact estimation",
    "ranking / scoring",
    "action hypothesis",
]
print("Methodology:", " -> ".join(methodology))


## 2. Target or proxy

The stages use different learning objects, so there is not one artificial label forced onto the whole pipeline.

### Clustering

Clustering is unsupervised and therefore has **no target column**. Its output is a performance archetype / peer group. Cluster statistics are fitted on training data only and are used to construct peer-relative features; they must not be calculated from the held-out future/test data.

### Supervised prediction

The final supervised target should be an **observed future outcome**, not a manually authored priority or action label.

The intended structure is:

**past feature window → decision point → non-overlapping future outcome window**

The primary later target is a future decline indicator such as `future_decline_30d`, defined from observed search performance after the decision point. A second continuous outcome can retain the magnitude of that future movement so that a small decline and a severe decline are not treated as equivalent.

The exact decline threshold, persistence requirement, and minimum-volume floor are deliberately **not invented here**. They will be fixed from the warehouse data contract before model training and sensitivity-tested. This notebook therefore specifies the target structure without pretending that an untested threshold is ground truth.

### Starter-data proxy

The 30,000-row starter CSV is only a trailing-90-day snapshot, so it cannot supply a genuinely future outcome. Its transparent current-window proxy is:

[
\text{decline\_proxy}=1
\quad \text{when} \quad
\texttt{trend\_direction = "down"}
]

The data dictionary defines `down` from the change in impressions between the most recent 30 days and the previous 30 days. This proxy is useful for demonstrating the framing, but it is a **defined current-window proxy**, not a future observed outcome and not proof that intervention is needed.

Because `trend_direction` is derived from `trend_pct`, neither `trend_direction` nor `trend_pct` may be used as predictive features when this proxy is the target.

### Ranking quantity

The final ranking should not learn a manually authored priority label. It should be derived from learned/observed quantities, conceptually:

[
\text{priority}
\propto
P(\text{future adverse state})
\times
E(\text{adverse movement magnitude})
\times
\text{measured exposure/value}
]

with ranking performed within an appropriate operational scope such as client. The exact normalization will be fixed later and validated against the chosen top-K metric rather than assigned arbitrary weights.

### Action outcome

The action engine does not claim to know the causal treatment effect from observational data. It maps the page's archetype, predicted future state, and strongest peer-relative abnormalities to a **recommended intervention hypothesis**. That recommendation can then be tested:

**automated recommendation → measurable intervention hypothesis → controlled or otherwise valid causal evaluation**.

In [ ]:
# Transparent starter-data proxy for Assignment 3 framing.
required_proxy_columns = {
    "content_id",
    "trend_direction",
    "trend_pct",
    "impressions_prev_30d",
    "impressions_last_30d",
}
missing = sorted(required_proxy_columns.difference(df.columns))
assert not missing, f"Missing required proxy columns: {missing}"

proxy_frame = df[[
    "content_id",
    "impressions_prev_30d",
    "impressions_last_30d",
    "trend_pct",
    "trend_direction",
]].copy()

proxy_frame["decline_proxy"] = (
    proxy_frame["trend_direction"].str.lower().eq("down").astype("int8")
)

print("Starter proxy: decline_proxy = 1 when trend_direction == 'down'.")
print("This is a current-window proxy, not a future causal or intervention label.\n")
print(proxy_frame["decline_proxy"].value_counts().sort_index())
print(f"Proxy positive rate: {proxy_frame['decline_proxy'].mean():.1%}\n")

display(proxy_frame.head(10))

print("Excluded from predictive features for this proxy: ['trend_direction', 'trend_pct']")

# Later warehouse target structure (not fabricated from this snapshot):
target_schema = pd.DataFrame({
    "field": [
        "future_decline_30d",
        "future_change_magnitude",
    ],
    "role": [
        "future-state classification target",
        "future-movement magnitude outcome",
    ],
    "available_in_starter_snapshot": [False, False],
})
display(target_schema)


## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.